# node_quantiles SPP smoke test

Integration check: does `NodeQuantileReaderVE` generate a correct SPP node-quantile table for a bid date?
Hits BigQuery/MySQL live (needs prod creds). Validates the table that feeds the quantile-risk dashboard
(`da_total_q*` / `rt_total_q*`).

In [1]:
import sys
sys.path.append("/var/www/python/Qingcheng/nighthawk/")
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 240)
from nighthawk.models.valuation.node_quantiles import NodeQuantileReaderVE
from nighthawk.models.securityselector import node_collector

OPEX     = "SPP"
BID_DATE = "2026-06-27"          # <-- change the date to test another day
QUANTS   = [1, 3, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 85, 90, 95, 97, 99]
QUANTILES = [q / 100 for q in QUANTS]
DA_COLS  = [f"da_total_q{q}" for q in QUANTS]
RT_COLS  = [f"rt_total_q{q}" for q in QUANTS]

## 1. Generate the table
Fetch the SPP Darwin nodes for the date (passing `node_list=[]` hits an empty `IN ()`), then call the reader.

In [2]:
nc = node_collector.VENodeCollector(opexchange=OPEX)
nodes_df = nc.get_darwin_nodes(start_dt=BID_DATE, end_dt=BID_DATE, hub_included=True)
node_list = sorted(nodes_df["node_num"].dropna().astype(int).unique().tolist())
print(f"Darwin {OPEX} nodes for {BID_DATE}: {len(node_list)}")
assert node_list, f"No Darwin {OPEX} nodes for {BID_DATE} - node selection not populated for that date."

reader = NodeQuantileReaderVE(OPEX)
df = reader.get_predicted_nodal_quantiles(
    BID_DATE, BID_DATE, node_list=node_list, quantiles=QUANTILES,
    price_type=["Total"], include_mean_preds=False)

# reorder quantile columns numerically (reader returns them string-sorted: q1,q10,q15,...)
df = df[["dt", "hr", "node_num"] + DA_COLS + RT_COLS]
print("shape:", df.shape)
df.head()

Darwin SPP nodes for 2026-06-27: 53
shape: (1272, 37)


,dt,hr,node_num,da_total_q1,da_total_q3,da_total_q5,da_total_q10,da_total_q15,da_total_q20,da_total_q30,da_total_q40,da_total_q50,da_total_q60,da_total_q70,da_total_q80,da_total_q85,da_total_q90,da_total_q95,da_total_q97,da_total_q99,rt_total_q1,rt_total_q3,rt_total_q5,rt_total_q10,rt_total_q15,rt_total_q20,rt_total_q30,rt_total_q40,rt_total_q50,rt_total_q60,rt_total_q70,rt_total_q80,rt_total_q85,rt_total_q90,rt_total_q95,rt_total_q97,rt_total_q99
0,2026-06-27,1,28,-9.338242,-0.540798,0.721691,3.538784,5.972656,8.164266,11.194543,13.715041,15.47149,17.129818,19.07865,20.451864,21.783264,23.015755,24.866369,25.770655,29.025156,-26.996309,-13.726187,-11.811671,-5.483448,-1.77015,-0.270187,3.705574,7.242614,11.043513,14.37727,18.134106,20.057873,21.452652,25.623922,39.059471,108.096191,221.766922
1,2026-06-27,1,29,1.001733,3.578134,3.401929,6.025773,7.211011,8.306218,8.755752,10.8595,12.516398,12.416738,14.352852,15.510423,15.995469,17.435526,17.926588,21.199202,26.550606,-23.503323,-10.975918,-10.17762,-5.533264,-1.593388,1.846691,4.883417,5.982483,7.398705,9.630281,12.878574,15.006572,20.224125,19.633286,29.062313,32.732418,71.544586
2,2026-06-27,1,45,5.294007,10.206294,11.573667,13.555537,15.233703,16.080057,16.12697,17.878132,20.016376,20.337166,22.713093,24.707695,26.02289,27.719267,30.353582,33.681698,39.535854,-15.222404,-7.172244,-3.890464,0.301252,3.601728,5.089563,7.968833,11.303679,14.562269,17.025291,20.49902,26.416424,31.076431,38.42271,60.26512,72.83502,121.1947
3,2026-06-27,1,61,5.706561,10.57355,11.882144,13.765267,15.449935,16.299116,16.303032,18.03479,20.171808,20.416513,22.772724,24.779213,26.112085,27.65209,30.210526,33.546143,39.32798,-14.807101,-6.969443,-3.703131,0.35239,3.709789,5.000013,7.824283,11.057742,14.299135,16.659538,19.93335,25.657444,30.114414,37.57116,59.30939,71.69073,119.89831
4,2026-06-27,1,66,7.827147,9.971639,10.984178,11.33628,11.823067,12.303365,12.716271,13.80547,14.959184,14.916926,16.328653,16.161564,16.246141,17.272072,18.954544,19.948978,18.564577,-3.011697,0.173345,3.116405,3.179118,6.868512,6.54737,8.88217,9.93482,11.985099,13.680768,16.224918,16.277122,19.160007,19.249275,22.015276,26.595404,34.126595


## 2. Validation checks
Critical: shape/date/columns/hours. Warnings: nulls & quantile monotonicity (the risk calc sorts quantiles, so crossing is tolerated downstream).

In [3]:
def monotonic_violations(cols):
    vals = df[cols].to_numpy(dtype="float64")
    diffs = vals[:, 1:] - vals[:, :-1]
    with np.errstate(invalid="ignore"):
        return int((diffs < -1e-6).any(axis=1).sum())

null_cells = int(df[DA_COLS + RT_COLS].isna().sum().sum())
hrs = df.groupby("node_num")["hr"].nunique()

checks = [
    ("non-empty",                   len(df) > 0,                         f"{len(df)} rows"),
    ("single requested dt",         set(df['dt'].astype(str)) == {BID_DATE}, f"{sorted(set(df['dt'].astype(str)))}"),
    ("has nodes",                   df['node_num'].nunique() > 0,        f"{df['node_num'].nunique()} nodes"),
    ("24 hours per node",           bool((hrs == 24).all()),             f"min={hrs.min()}, max={hrs.max()}"),
    ("no nulls in quantile cols",   null_cells == 0,                     f"{null_cells} null cells"),
    ("DA quantiles non-decreasing", monotonic_violations(DA_COLS) == 0,  f"{monotonic_violations(DA_COLS)} rows violate"),
    ("RT quantiles non-decreasing", monotonic_violations(RT_COLS) == 0,  f"{monotonic_violations(RT_COLS)} rows violate"),
]
for name, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name:32s} {detail}")

[PASS] non-empty                        1272 rows
[PASS] single requested dt              ['2026-06-27']
[PASS] has nodes                        53 nodes
[PASS] 24 hours per node                min=24, max=24
[FAIL] no nulls in quantile cols        408 null cells
[FAIL] DA quantiles non-decreasing      810 rows violate
[FAIL] RT quantiles non-decreasing      655 rows violate


## 3. Inspect the table
Readable sample, where the nulls are, and save the full table.

In [4]:
view = ["node_num", "hr",
        "da_total_q1", "da_total_q5", "da_total_q50", "da_total_q95", "da_total_q99",
        "rt_total_q1", "rt_total_q5", "rt_total_q50", "rt_total_q95", "rt_total_q99"]
display(df[df["hr"] == 1][view].head(10))

# where are the nulls?
nulls_by_col = df[DA_COLS + RT_COLS].isna().sum()
print("null cells by column (nonzero):")
print(nulls_by_col[nulls_by_col > 0] if (nulls_by_col > 0).any() else "  none")
null_nodes = df.loc[df[DA_COLS + RT_COLS].isna().any(axis=1), "node_num"].unique()
print("nodes with >=1 null:", sorted(int(n) for n in null_nodes))

# save the full table
out = f"node_quantiles_{OPEX}_{BID_DATE}.csv"
df.to_csv(out, index=False)
print("saved:", out)

,node_num,hr,da_total_q1,da_total_q5,da_total_q50,da_total_q95,da_total_q99,rt_total_q1,rt_total_q5,rt_total_q50,rt_total_q95,rt_total_q99
0,28,1,-9.338242,0.721691,15.47149,24.866369,29.025156,-26.996309,-11.811671,11.043513,39.059471,221.766922
1,29,1,1.001733,3.401929,12.516398,17.926588,26.550606,-23.503323,-10.17762,7.398705,29.062313,71.544586
2,45,1,5.294007,11.573667,20.016376,30.353582,39.535854,-15.222404,-3.890464,14.562269,60.26512,121.1947
3,61,1,5.706561,11.882144,20.171808,30.210526,39.32798,-14.807101,-3.703131,14.299135,59.30939,119.89831
4,66,1,7.827147,10.984178,14.959184,18.954544,18.564577,-3.011697,3.116405,11.985099,22.015276,34.126595
5,104,1,8.297854,11.258783,15.171116,19.045176,18.63267,-2.324362,3.660468,12.279842,22.040052,33.575584
6,185,1,3.157627,2.888723,12.196284,18.678982,25.989141,-11.276816,2.31961,14.873968,38.499893,63.605057
7,186,1,3.530489,3.169651,12.443974,18.8533,26.018421,-10.748153,2.70502,15.114775,38.478504,63.367462
8,211,1,0.110938,10.674066,19.769154,33.510536,44.04398,-17.150373,-0.892121,17.271872,50.846802,135.867996
9,313,1,-5.645894,-0.255982,14.571538,30.91955,41.41441,-17.372471,-2.54518,12.703018,64.417107,173.63559


null cells by column (nonzero):
da_total_q1     12
da_total_q3     12
da_total_q5     12
da_total_q10    12
da_total_q15    12
da_total_q20    12
da_total_q30    12
da_total_q40    12
da_total_q50    12
da_total_q60    12
da_total_q70    12
da_total_q80    12
da_total_q85    12
da_total_q90    12
da_total_q95    12
da_total_q97    12
da_total_q99    12
rt_total_q1     12
rt_total_q3     12
rt_total_q5     12
rt_total_q10    12
rt_total_q15    12
rt_total_q20    12
rt_total_q30    12
rt_total_q40    12
rt_total_q50    12
rt_total_q60    12
rt_total_q70    12
rt_total_q80    12
rt_total_q85    12
rt_total_q90    12
rt_total_q95    12
rt_total_q97    12
rt_total_q99    12
dtype: int64
nodes with >=1 null: [349, 542, 546, 550, 601, 635, 636, 704, 734, 735, 766, 792]
saved: node_quantiles_SPP_2026-06-27.csv


## 4. Why `get_expected_portfolio_performance` can miss node quantiles

`get_expected_portfolio_performance` evaluates the **portfolio's** nodes (`node_list = df['node_num'].unique()`)
and **left-merges** the predicted quantiles. So:
- portfolio nodes **not** covered by Darwin/Curie/Watson (nodal+hub+default) -> NaN quantiles -> NaN risk
- the final result **drops the raw `*_total_q*` columns** (it returns only `quantileN_risk` / `exp_*`)

This cell loads the real SPP portfolio, checks node coverage, and runs the evaluation.

In [10]:
from nighthawk.data.product.ve import DailyBidsManager
from nighthawk.models.evaluation.ve_portfolio_evaluation import VEPortfolioEvaluation

# 1) load the current SPP portfolio (bids) for the date
bm = DailyBidsManager(OPEX, BID_DATE)
portfolio = bm.get_bids_from_table(label='current')
print("portfolio rows:", len(portfolio))
assert len(portfolio), f"No bids for {OPEX} {BID_DATE}"
port_nodes = sorted(portfolio['node_num'].dropna().astype(int).unique().tolist())
print("portfolio nodes:", len(port_nodes))

# 2) which portfolio nodes does get_predicted_nodal_quantiles actually cover?
q = reader.get_predicted_nodal_quantiles(BID_DATE, BID_DATE, node_list=port_nodes,
                                         quantiles=QUANTILES, price_type=['Total'])
covered = sorted(set(q.loc[q[RT_COLS].notna().any(axis=1), 'node_num'].astype(int))) if len(q) else []
missing = sorted(set(port_nodes) - set(covered))
print("nodes with >=1 non-null rt quantile:", len(covered))
print("portfolio nodes MISSING quantiles :", len(missing), missing[:40])

portfolio rows: 2429
portfolio nodes: 35
nodes with >=1 non-null rt quantile: 35
portfolio nodes MISSING quantiles : 0 []


In [9]:
# 3) run the evaluation and see what it returns
pe  = VEPortfolioEvaluation(OPEX, portfolio)
res = pe.get_expected_portfolio_performance(quantiles=QUANTILES)

print("result rows:", len(res))
print("result keeps RAW rt_total_q* cols?:", any(c.startswith('rt_total_q') for c in res.columns))
print("result columns:", list(res.columns))

# which nodes ended up WITHOUT a quantile risk value (because their quantiles were NaN)?
if 'quantile1_risk' in res.columns:
    by_node = res.groupby('node_num')['quantile1_risk'].apply(lambda s: s.notna().any())
    print("nodes WITH quantile1_risk   :", int(by_node.sum()), "/", len(by_node))
    print("nodes WITHOUT quantile1_risk:", sorted(int(n) for n in by_node[~by_node].index)[:40])
res.head()

result rows: 2429
result keeps RAW rt_total_q* cols?: False
result columns: ['dt', 'hr', 'segment', 'node_num', 'bid_mw', 'bid_price', 'strategy', 'incdec', 'exp_total_profit', 'exp_clear_mw', 'quantile1_risk', 'quantile3_risk', 'quantile5_risk', 'quantile10_risk', 'quantile15_risk', 'quantile20_risk', 'quantile30_risk', 'quantile40_risk', 'quantile50_risk', 'quantile60_risk', 'quantile70_risk', 'quantile80_risk', 'quantile85_risk', 'quantile90_risk', 'quantile95_risk', 'quantile97_risk', 'quantile99_risk', 'exp_profit_per_bid_mw', 'exp_roq5']
nodes WITH quantile1_risk   : 35 / 35
nodes WITHOUT quantile1_risk: []


,dt,hr,segment,node_num,bid_mw,bid_price,strategy,incdec,exp_total_profit,exp_clear_mw,quantile1_risk,quantile3_risk,quantile5_risk,quantile10_risk,quantile15_risk,quantile20_risk,quantile30_risk,quantile40_risk,quantile50_risk,quantile60_risk,quantile70_risk,quantile80_risk,quantile85_risk,quantile90_risk,quantile95_risk,quantile97_risk,quantile99_risk,exp_profit_per_bid_mw,exp_roq5
0,2026-06-27,1,1,28,1.9,11.2,Darwin,Decrement,1.1,0.6,-73.9,-48.2,-44.5,-32.3,-25.1,-22.2,-14.5,-7.7,-0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.0
1,2026-06-27,1,2,28,1.2,11.2,HighCap,Decrement,0.7,0.3,-44.3,-28.9,-26.7,-19.4,-15.1,-13.3,-8.7,-4.6,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.0
2,2026-06-27,1,3,28,1.9,8.2,Darwin,Decrement,0.6,0.4,-68.1,-42.4,-38.7,-26.5,-19.3,-16.4,-8.7,-1.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0
3,2026-06-27,1,4,28,1.2,8.2,HighCap,Decrement,0.4,0.2,-40.9,-25.5,-23.2,-15.9,-11.6,-9.8,-5.2,-1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0
4,2026-06-27,1,1,211,1.9,15.4,Darwin,Decrement,0.3,0.4,-63.0,-37.6,-31.5,-21.3,-16.5,-12.6,-6.7,-1.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0


## 5. Combined table — which source supplied each node

The reader fills the combined table **cell-by-cell** in priority order **Darwin > Curie > Watson > Fourier > hub/default**.
This cell probes a representative column (`rt_total_q50`) and labels each `(node, hr)` by the first source that covered it,
so you can see how much the newly-added Fourier fallback actually contributes vs. the older strategies.

(Probe is one column; a node counted under e.g. Fourier here can still be NaN in *other* quantile columns — see §2/§3 for full-column null counts.)

In [11]:
# 5) Combined table: which SOURCE supplied each node's quantile?
KEY, PROBE = ['dt', 'hr', 'node_num'], 'rt_total_q50'

def _covered_keys(method=None, fourier=False):
    if fourier:
        s = reader.get_predicted_nodal_quantiles_fourier_dnn(
            BID_DATE, BID_DATE, quantiles=QUANTILES, node_list=node_list)
    else:
        s = method(BID_DATE, BID_DATE, node_list=node_list, quantiles=QUANTILES,
                   hub_only=False, node_only=True, price_type=['Total'])
    s = s.loc[s[PROBE].notna(), KEY].drop_duplicates()   # dedup-safe coverage by presence
    return set(map(tuple, s.to_numpy()))

cov = {
    'Darwin':  _covered_keys(reader.get_predicted_nodal_quantiles_darwin),
    'Curie':   _covered_keys(reader.get_predicted_nodal_quantiles_curie),
    'Watson':  _covered_keys(reader.get_predicted_nodal_quantiles_watson),
    'Fourier': _covered_keys(fourier=True),
}

comb = df[KEY + [PROBE]].copy()
def _attr(k, v):
    for name in ['Darwin', 'Curie', 'Watson', 'Fourier']:   # priority order
        if k in cov[name]:
            return name
    return 'hub/default' if pd.notna(v) else 'none'
comb['source'] = [_attr(tuple(k), v) for k, v in zip(comb[KEY].to_numpy(), comb[PROBE])]

print(f"rows: {len(comb)}  (nodes*24 = {df['node_num'].nunique() * 24})")
print("\nper node-hour cell  (probe = rt_total_q50):")
print(comb['source'].value_counts().to_string())
print("\nper node, dominant source:")
dom = comb.groupby('node_num')['source'].agg(lambda s: s.value_counts().index[0])
print(dom.value_counts().to_string())

# combined sample (hr 1) with the winning source labelled
comb[comb['hr'] == 1].head(15)

rows: 1272  (nodes*24 = 1272)

per node-hour cell  (probe = rt_total_q50):
source
Fourier        1176
Darwin           72
hub/default      24

per node, dominant source:
source
Fourier        49
Darwin          3
hub/default     1


,dt,hr,node_num,rt_total_q50,source
0,2026-06-27,1,28,11.043513,Darwin
1,2026-06-27,1,29,7.398705,Fourier
2,2026-06-27,1,45,14.562269,Fourier
3,2026-06-27,1,61,14.299135,Fourier
4,2026-06-27,1,66,11.985099,Fourier
5,2026-06-27,1,104,12.279842,Fourier
6,2026-06-27,1,185,14.873968,Fourier
7,2026-06-27,1,186,15.114775,Fourier
8,2026-06-27,1,211,17.271872,Darwin
9,2026-06-27,1,313,12.703018,Darwin
